In [3]:
import os
import sys
import shutil  # 🧹 مكتبة إدارة الملفات لحذف الـ Checkpoint
from datetime import datetime, timezone

# 1. إعداد مسارات المكتبات لتعمل أوفلاين
offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

internal_ivy_path = "/home/jovyan/.ivy2/jars/*"

# 2. بناء جلسة سبارك المتزنة
spark = SparkSession.builder \
    .appName("Kafka_To_Raw_Bronze_Incremental") \
    .config("spark.jars", internal_ivy_path) \
    .config("spark.sql.streaming.minBatchesToRetain", "30") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .config("spark.sql.streaming.fileSink.log.cleanupDelay", "60000") \
    .getOrCreate()

try:
    print("📡 [الطبقة البرونزية] جاري الاتصال بكافكا وسحب البيانات الخام...")
    
    # تحويل القراءة إلى نظام البث المقيد بالدفعة الحالية لضمان تتبع الـ Checkpoint
    kafka_stream_df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "smarthome-kafka:29092") \
        .option("subscribe", "energy_events") \
        .option("startingOffsets", "earliest") \
        .option("failOnDataLoss", "false") \
        .load()
        
    raw_df = kafka_stream_df.selectExpr("CAST(value AS STRING) as json_payload")

    # 3. هيكل الـ JSON الكامل والمطابق لبيانات المحاكي
    data_schema = StructType([
        StructField("timestamp", StringType(), True),
        StructField("house_type", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("zone", StringType(), True),
        StructField("device_id", StringType(), True),
        StructField("device_type", StringType(), True),
        StructField("is_room_occupied", BooleanType(), True), 
        StructField("power_consumption_watts", DoubleType(), True),
        StructField("status", StringType(), True)
    ])

    # 4. طبقة فك التشفير فقط (تحويل الـ JSON إلى أعمدة دون أي فلترة أو تنظيف)
    # لاحظ هنا: تم إلغاء خطوة الفلترة (filter) تماماً لنحتفظ بالبيانات المتسخة والـ Nulls والـ SENSOR_FAULT
    raw_parsed_df = raw_df.withColumn("data", from_json(col("json_payload"), data_schema)) \
                          .select("data.*")

    # 5. تحديد مسارات التخزين والـ Checkpoint للطبقة البرونزية الخام (Raw/Dirty)
    raw_parquet_path = "/home/jovyan/work/storage/raw_bronze_parquet"
    checkpoint_raw_path = "/home/jovyan/work/storage/checkp_raw_parquet_increment_auto"

    # 🧹 تنظيف الـ Checkpoint القديم لتفادي البطء
    if os.path.exists(checkpoint_raw_path):
        try:
            print("🧹 جاري تنظيف مجلد الـ Checkpoint البرونزي القديم لتسريع التشغيل...")
            shutil.rmtree(checkpoint_raw_path)
            print("✅ تم تصفير الـ Checkpoint بنجاح.")
        except Exception as e:
            print(f"⚠️ تنبيه: لم نتمكن من حذف مجلد الـ Checkpoint، قد يكون قيد الاستخدام: {e}")

    print("💾 جاري حفظ البيانات الخام (المتسخة) بنمط تزايدي (Append) في الطبقة البرونزية...")
    
    # تشغيل عملية الحفظ التزايدي باستخدام availableNow=True ليعمل كـ Batch
    query = raw_parsed_df.writeStream \
        .format("parquet") \
        .outputMode("append") \
        .partitionBy("zone") \
        .option("path", raw_parquet_path) \
        .option("checkpointLocation", checkpoint_raw_path) \
        .trigger(availableNow=True) \
        .start()
        
    query.awaitTermination()
    print("✅ [نجاح] تم جلب وحفظ البيانات الخام بنجاح في مجلد raw_bronze_parquet!")

except Exception as e:
    print(f"❌ فشل في محرك حفظ البيانات الخام: {e}")

finally:
    spark.stop()

📡 [الطبقة البرونزية] جاري الاتصال بكافكا وسحب البيانات الخام...
🧹 جاري تنظيف مجلد الـ Checkpoint البرونزي القديم لتسريع التشغيل...
✅ تم تصفير الـ Checkpoint بنجاح.
💾 جاري حفظ البيانات الخام (المتسخة) بنمط تزايدي (Append) في الطبقة البرونزية...
✅ [نجاح] تم جلب وحفظ البيانات الخام بنجاح في مجلد raw_bronze_parquet!
